# 1. 초기 세팅


In [3]:
from sqlalchemy import create_engine, Column, Integer, String, Float, ForeignKey
from sqlalchemy.orm import relationship, sessionmaker, declarative_base

# 1. 엔진 연결 (기존에 만든 db 파일명 확인)
engine = create_engine('sqlite:///crime_data.db') 
Session = sessionmaker(bind=engine)
session = Session()
Base = declarative_base()

# 2. 테이블

In [4]:
# --- [ 마스터 테이블 ] ---

class RegionMaster(Base):
    """지역 마스터 (예: 서울 강남구, 서울 종로구)"""
    __tablename__ = 'region_master'
    id = Column(Integer, primary_key=True, autoincrement=True)
    region_name = Column(String, unique=True, nullable=False)

    # 관계 설정
    mappers = relationship("RegionMapper", back_populates="region")
    region_crimes = relationship("CrimeRegion", back_populates="region")

class CrimeCategory(Base):
    """범죄 종류 마스터 (예: 절도, 폭행)"""
    __tablename__ = 'crime_category'
    id = Column(Integer, primary_key=True, autoincrement=True)
    main_cat = Column(String)  # 범죄대분류
    sub_cat = Column(String, unique=True) # 범죄중분류

    # 관계 설정
    region_stats = relationship("CrimeRegion", back_populates="category")
    time_stats = relationship("CrimeTime", back_populates="category")
    week_stats = relationship("CrimeWeek", back_populates="category")

# --- [ 매핑 테이블 (핵심 연결 고리) ] ---

class RegionMapper(Base):
    """핫스팟과 지역 마스터를 잇는 중간 다리 (mapping_fix 기반)"""
    __tablename__ = 'region_mapper'
    id = Column(Integer, primary_key=True, autoincrement=True)
    AREA_GU = Column(String)
    CATEGORY = Column(String)
    NO = Column(Integer)
    AREA_CD = Column(String)
    AREA_NM = Column(String)
    ENG_NM = Column(String)
    
    # FK 설정
    hotspot_id = Column(Integer, ForeignKey('hotspot_api.id'))
    region_id = Column(Integer, ForeignKey('region_master.id'))

    # 관계 설정
    hotspot = relationship("HotspotAPI", back_populates="mapper")
    region = relationship("RegionMaster", back_populates="mappers")

# --- [ 데이터 테이블 ] ---

class HotspotAPI(Base):
    """핫스팟 API 정보 (seoul live cache 기반)"""
    __tablename__ = 'hotspot_api'
    id = Column(Integer, primary_key=True, autoincrement=True)
    area_name = Column(String)
    congest_lvl = Column(Integer)
    ppltn_min = Column(Integer)
    ppltn_max = Column(Integer)
    temp = Column(Float)
    update_time = Column(String)
    collected_at = Column(String)

    # 매핑 테이블과 1:1 연결
    mapper = relationship("RegionMapper", back_populates="hotspot", uselist=False)

class CrimeRegion(Base):
    """경찰청 지역별 범죄 통계 (police_region_fix 기반)"""
    __tablename__ = 'crime_region'
    id = Column(Integer, primary_key=True, autoincrement=True)
    crime_count = Column(Integer)
    
    region_id = Column(Integer, ForeignKey('region_master.id'))
    category_id = Column(Integer, ForeignKey('crime_category.id'))
    
    region = relationship("RegionMaster", back_populates="region_crimes")
    category = relationship("CrimeCategory", back_populates="region_stats")

class CrimeTime(Base):
    """시간대별 범죄 통계 (전국 통계용)"""
    __tablename__ = 'crime_time'
    id = Column(Integer, primary_key=True, autoincrement=True)
    time_range = Column(String)
    crime_count = Column(Float)
    
    category_id = Column(Integer, ForeignKey('crime_category.id'))
    category = relationship("CrimeCategory", back_populates="time_stats")

class CrimeWeek(Base):
    """요일별 범죄 통계 (전국 통계용)"""
    __tablename__ = 'crime_week'
    id = Column(Integer, primary_key=True, autoincrement=True)
    day_of_week = Column(String)
    crime_count = Column(Integer)
    
    category_id = Column(Integer, ForeignKey('crime_category.id'))
    category = relationship("CrimeCategory", back_populates="week_stats")

In [6]:
# # 2. 테이블 생성
# Base.metadata.drop_all(engine)
# Base.metadata.create_all(engine)
# print("의도하신 매핑 구조로 DB 생성 완료!")

In [7]:
from sqlalchemy import func

# 핫스팟 이름으로 지역 찾기 및 해당 지역의 범죄 건수 합산
result = session.query(func.sum(CrimeRegion.crime_count))\
    .join(RegionMaster, CrimeRegion.region_id == RegionMaster.id)\
    .join(RegionMapper, RegionMaster.id == RegionMapper.region_id)\
    .join(HotspotAPI, RegionMapper.hotspot_id == HotspotAPI.id)\
    .filter(HotspotAPI.area_name == '강남 MICE 관광특구')\
    .scalar()

print(f"해당 핫스팟 소속 구의 총 범죄 발생 건수: {result}건")

해당 핫스팟 소속 구의 총 범죄 발생 건수: 5599건


1. 모든 핫스팟의 이름과 현재 혼잡도 등급 조회하기
목표: HotspotAPI 테이블의 기본 컬럼 조회 연습.

In [10]:
hotsop_api=session.query(HotspotAPI).limit(5).all()

for row in hotsop_api:
    print(row.area_name, row.congest_lvl)

강남 MICE 관광특구 약간 붐빔
동대문 관광특구 약간 붐빔
명동 관광특구 약간 붐빔
이태원 관광특구 약간 붐빔
잠실 관광특구 여유


2. '서울 강남구' 지역 마스터의 ID(PK) 찾기
목표: filter_by 또는 filter를 사용한 단일 행 조회.

In [15]:
gangnam = session.query(RegionMaster).filter_by(region_name='서울 강남구').all()

gangnam[0].id


23

3. 범죄 중분류(sub_cat)가 '절도'인 카테고리의 모든 정보 조회하기
목표: 특정 조건을 만족하는 마스터 테이블 조회.

In [17]:
jeoldo = session.query(CrimeCategory).filter_by(sub_cat='절도').all()
print(jeoldo[0].id, jeoldo[0].main_cat, jeoldo[0].sub_cat)

5 절도범죄 절도


4. 실시간 인구 최대값(ppltn_max)이 50,000명 이상인 핫스팟들 나열하기
목표: 숫자형 컬럼의 비교 연산자 사용.

In [18]:
five_pp = session.query(HotspotAPI).filter(HotspotAPI.ppltn_max>=50000).all()

for row in five_pp:
    print(row.area_name)

명동 관광특구
잠실 관광특구
종로·청계 관광특구
홍대 관광특구
광화문·덕수궁
강남역
선릉역
역삼역
오목교역·목동운동장
여의도
잠실롯데타워 일대


5. 각 핫스팟이 어느 지역구(region_name)에 속해 있는지 함께 출력하기
목표: HotspotAPI - RegionMapper - RegionMaster 3개 테이블 조인.

In [19]:
hotspot_region = session.query(RegionMapper).limit(10).all()
for row in hotspot_region:
    print(row.AREA_GU, row.AREA_NM)

서울 강남구 강남 MICE 관광특구
서울 중구 동대문 관광특구
서울 중구 명동 관광특구
서울 용산구 이태원 관광특구
서울 송파구 잠실 관광특구
서울 종로구 종로·청계 관광특구
서울 마포구 홍대 관광특구
서울 종로구 경복궁
서울 중구 광화문·덕수궁
서울 종로구 광화문·덕수궁


6. '서울 종로구'에 해당하는 모든 핫스팟(area_name) 목록 가져오기
목표: 지역명을 조건으로 연결된 핫스팟 찾기.

In [21]:
jongro_hot = session.query(RegionMapper).filter_by(region_id=1).all()
for row in jongro_hot:
    print(row.AREA_GU, row.AREA_NM)

서울 종로구 종로·청계 관광특구
서울 종로구 경복궁
서울 종로구 광화문·덕수궁
서울 종로구 보신각
서울 종로구 창덕궁·종묘
서울 종로구 동대문역
서울 종로구 혜화역
서울 종로구 광장(전통)시장
서울 종로구 북촌한옥마을
서울 종로구 서촌
서울 종로구 인사동
서울 종로구 광화문광장
서울 종로구 청와대
서울 종로구 익선동


7. 전국 통계 데이터에서 '월요일'에 발생하는 모든 범죄의 종류와 건수 조회하기
목표: CrimeWeek와 CrimeCategory 조인 조회

In [30]:
monday = session.query(CrimeWeek)\
    .join(CrimeCategory, CrimeWeek.category_id == CrimeCategory.id)\
    .filter(CrimeWeek.day_of_week=='월').all()

for row in monday:
    print(row.day_of_week, row.category.main_cat)

월 강력범죄
월 강력범죄
월 강력범죄
월 강력범죄
월 마약범죄
월 절도범죄
월 폭력범죄
월 폭력범죄
월 폭력범죄


8. 각 지역구(region_name)별로 등록된 범죄 통계의 총 건수(sum) 계산하기
목표: sqlalchemy.func.sum과 group_by 활용.

In [35]:
region_stat = session.query(RegionMaster.region_name, func.sum(CrimeRegion.crime_count))\
    .join(RegionMaster, CrimeRegion.region_id == RegionMaster.id)\
    .join(CrimeCategory, CrimeRegion.category_id == CrimeCategory.id)\
    .group_by(CrimeRegion.region_id).all()

region_stat

[('서울 종로구', 2314),
 ('서울 중구', 2584),
 ('서울 용산구', 2752),
 ('서울 성동구', 1809),
 ('서울 광진구', 2372),
 ('서울 동대문구', 2870),
 ('서울 중랑구', 2682),
 ('서울 성북구', 1804),
 ('서울 강북구', 1981),
 ('서울 도봉구', 1505),
 ('서울 노원구', 2738),
 ('서울 은평구', 2723),
 ('서울 서대문구', 1759),
 ('서울 마포구', 3200),
 ('서울 양천구', 2573),
 ('서울 강서구', 3440),
 ('서울 구로구', 2935),
 ('서울 금천구', 1758),
 ('서울 영등포구', 3526),
 ('서울 동작구', 2112),
 ('서울 관악구', 3907),
 ('서울 서초구', 3218),
 ('서울 강남구', 5599),
 ('서울 송파구', 4519),
 ('서울 강동구', 2938),
 ('경기도 과천시', 476)]

9. 시간대(time_range)가 '00:00-02:59'인 데이터만 뽑아서 범죄 건수가 높은 순(DESC)으로 정렬하기
목표: order_by를 활용한 정렬 연습.

10. 현재 '약간 붐빔' 이상의 혼잡도를 가진 핫스팟들이 속한 구의 이름(중복 제거) 출력하기
목표: distinct()를 사용한 중복 제거 조회.